# Notebook 2: Simulation Analysis
**Paper:** *Agents for Agents: An Interrogator-Based Secure Framework for Autonomous IoUT*

This notebook:
1. Runs the IoUT multi-agent simulation (5 runs for Colab speed)
2. Compares the proposed framework vs. static trust and Bayesian baselines
3. Reproduces Figures 4, 5, and 6 from the paper
4. Prints the summary statistics table

---
> **Note:** The paper reports results averaged over 30 runs. This demo uses 5 for speed.

In [ ]:
# ── Cell 1: Install and setup ─────────────────────────────────────────────
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'matplotlib', 'tqdm'], check=True)
if not os.path.exists('IoUT-Interrogator-Framework'):
    subprocess.run(['git', 'clone',
                    'https://github.com/aliakarma/IoUT-Interrogator-Framework.git'], check=True)
os.chdir('IoUT-Interrogator-Framework')
sys.path.insert(0, '.')
print('Ready. CWD:', os.getcwd())

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from simulation.scripts.environment import IoUTEnvironment
from simulation.scripts.run_simulation import aggregate_runs
print('Imports OK.')

In [ ]:
# ── Cell 3: Run simulation (5 runs for Colab demo) ────────────────────────
NUM_RUNS      = 5    # Set to 30 to reproduce paper results
NUM_INTERVALS = 20
BASE_SEED     = 42
CONFIG_PATH   = 'simulation/configs/simulation_params.json'

all_results = []
for run_idx in tqdm(range(NUM_RUNS), desc='Simulation runs'):
    env = IoUTEnvironment(config_path=CONFIG_PATH, seed=BASE_SEED + run_idx)
    result = env.run(num_intervals=NUM_INTERVALS)
    all_results.append(result)

agg_df = aggregate_runs(all_results)
print(f'\nSimulation complete. Shape: {agg_df.shape}')
agg_df.head(3)

In [ ]:
# ── Cell 4: Plot accuracy comparison ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
x = agg_df['interval']

for col, label, color, ls in [
    ('accuracy_proposed', 'Proposed Interrogator Framework', '#2ca02c', '-'),
    ('accuracy_bayesian', 'Bayesian Trust',                  '#ff7f0e', '-.'),
    ('accuracy_static',   'Static Trust',                    '#1f77b4', '--'),
]:
    mean = agg_df[f'{col}_mean']
    std  = agg_df[f'{col}_std']
    ax.plot(x, mean, label=label, color=color, linestyle=ls, linewidth=2)
    ax.fill_between(x, mean - std, mean + std, alpha=0.12, color=color)

ax.set_xlabel('Monitoring Intervals', fontsize=12)
ax.set_ylabel('Anomaly Detection Accuracy (%)', fontsize=12)
ax.set_title('Anomaly Detection Accuracy — Simulation Results', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('analysis/plots', exist_ok=True)
plt.savefig('analysis/plots/trust_accuracy_sim.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Plot energy comparison ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for col, label, color, ls in [
    ('energy_static',   'Static Trust',                    '#1f77b4', '-'),
    ('energy_bayesian', 'Bayesian Trust',                  '#ff7f0e', '--'),
    ('energy_proposed', 'Proposed Interrogator Framework', '#2ca02c', '-'),
]:
    mean = agg_df[f'{col}_mean']
    std  = agg_df[f'{col}_std']
    ax.plot(x, mean, label=label, color=color, linestyle=ls, linewidth=2)
    ax.fill_between(x, mean - std, mean + std, alpha=0.12, color=color)

ax.set_xlabel('Time (Monitoring Intervals)', fontsize=12)
ax.set_ylabel('Residual Energy (%)', fontsize=12)
ax.set_title('Residual Energy — Simulation Results', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('analysis/plots/energy_comparison_sim.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 6: Plot PDR comparison ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for col, label, color, ls in [
    ('pdr_proposed', 'Proposed Interrogator Framework', '#2ca02c', '-'),
    ('pdr_bayesian', 'Bayesian Trust',                  '#ff7f0e', '-.'),
    ('pdr_static',   'Static Trust',                    '#1f77b4', '--'),
]:
    mean = agg_df[f'{col}_mean']
    std  = agg_df[f'{col}_std']
    ax.plot(x, mean, label=label, color=color, linestyle=ls, linewidth=2)
    ax.fill_between(x, mean - std, mean + std, alpha=0.12, color=color)

ax.set_xlabel('Monitoring Intervals', fontsize=12)
ax.set_ylabel('Packet Delivery Ratio (%)', fontsize=12)
ax.set_title('Packet Delivery Ratio — Simulation Results', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('analysis/plots/pdr_comparison_sim.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 7: Summary statistics table ──────────────────────────────────────
summary = pd.DataFrame({
    'Metric': [
        'Accuracy — Proposed (%)', 'Accuracy — Bayesian (%)', 'Accuracy — Static (%)',
        'PDR — Proposed (%)',      'PDR — Bayesian (%)',       'PDR — Static (%)',
    ],
    'Mean (paper)': [94.2, 86.1, 72.5, 91.6, 86.7, 79.4],
    'Mean (sim)': [
        agg_df['accuracy_proposed_mean'].mean(),
        agg_df['accuracy_bayesian_mean'].mean(),
        agg_df['accuracy_static_mean'].mean(),
        agg_df['pdr_proposed_mean'].mean(),
        agg_df['pdr_bayesian_mean'].mean(),
        agg_df['pdr_static_mean'].mean(),
    ],
})
summary['Mean (sim)'] = summary['Mean (sim)'].round(1)
print(summary.to_string(index=False))

os.makedirs('analysis/stats', exist_ok=True)
summary.to_csv('analysis/stats/summary_table.csv', index=False)
print('\nSaved to analysis/stats/summary_table.csv')

In [ ]:
# ── Cell 8: Verify v3 energy fix — three independent depletion curves ──
print('=== v3 Energy Model Verification ===')
print(f'Final residuals at interval {agg_df["interval"].iloc[-1]}:')
print(f'  Static:   {agg_df["energy_static_mean"].iloc[-1]:.1f}%')
print(f'  Bayesian: {agg_df["energy_bayesian_mean"].iloc[-1]:.1f}%')
print(f'  Proposed: {agg_df["energy_proposed_mean"].iloc[-1]:.1f}%')

drain_s = 100 - agg_df['energy_static_mean'].iloc[-1]
drain_p = 100 - agg_df['energy_proposed_mean'].iloc[-1]
overhead = (drain_p - drain_s) / drain_s * 100
print(f'  Proposed overhead vs static: {overhead:.1f}%  (paper: 5.8%)')

# v1 bug: all methods showed 100% residual (no visible depletion)
# v3 fix: drain_rate calibrated to paper Figure 5
assert agg_df['energy_proposed_mean'].iloc[-1] < 95, \
    'Energy still at 100% — v1 energy bug not fixed'
assert agg_df['energy_static_mean'].iloc[-1] > \
       agg_df['energy_proposed_mean'].iloc[-1], \
    'Static should retain more energy than proposed'
print('v3 energy model PASSED')